In [1]:
import sys
from pathlib import Path

# Asumsikan kamu menjalankan notebook dari 'data_pipeline_pyspark/notebooks'
# dan kamu ingin import dari 'data_pipeline_pyspark/src'
project_root = Path.cwd().parent
sys.path.append(str(project_root))
import pandas as pd

from src.utils.helper import startup_investments_engine_pyspark

from src.staging.extract.extract_db import extract_database
from src.staging.extract.extract_db_pyspark import extract_database as extract_database_pyspark
from src.staging.extract.extract_spreadsheet_pyspark import extract_sheet_spark,extract_spreadsheet as extract_spreadsheet_pyspark
from src.staging.extract.extract_api_pyspark import extract_api_milestones_spark,extract_api_spark,extract_backfilling_spark,extract_api_milestones_spark

from src.staging.extract.extract_spreadsheet import extract_spreadsheet
from src.staging.extract.extract_api import extract_api_milestones,extract_backfilling
from src.staging.load.load import load_staging
from src.staging.extract.extract_spreadsheet import extract_spreadsheet, extract_sheet

from src.warehouse.extract.extract_db import extract_database as extract_staging
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp
from src.staging.load.load_pyspark import load_staging_pyspark_upsert
from datetime import datetime
from src.warehouse.extract.extract_db_pyspark import extract_database as extract_db_pyspark_staging

from src.warehouse.extract.extract_db_pyspark import extract_database as extract_db_pyspark_staging

from src.warehouse.transform.dim_company_pyspark import transform_dim_company_spark
from src.warehouse.transform.dim_people_pyspark import transform_dim_people_spark
from src.warehouse.transform.dim_relationship_pyspark import transform_dim_relationship_spark
from src.warehouse.transform.fact_acquisitions_pyspark import transform_fact_acquisitions_spark
from src.warehouse.transform.fact_acquisitions_pyspark import transform_fact_acquisitions_spark
from src.warehouse.transform.fact_funding_rounds_pyspark import transform_fact_funding_rounds_spark
from src.warehouse.transform.fact_funds_pyspark import transform_fact_funds_spark
from src.warehouse.transform.fact_ipos_pyspark import transform_fact_ipos_spark
from src.warehouse.transform.fact_milestones_pyspark import transform_fact_milestones_spark
from src.warehouse.transform.fact_investments_pyspark import transform_fact_investments_spark
from src.warehouse.load.load_pyspark import load_warehouse_pyspark_upsert

# Declare spark

In [2]:
from pyspark.sql import SparkSession

# Membuat SparkSession dengan semua core lokal dan paksa port 4041
spark = SparkSession.builder \
    .appName("Cek Spark UI") \
    .config("spark.ui.enabled", "true") \
    .config("spark.ui.port", "4040") \
    .master("local[*]") \
    .getOrCreate()

print(f"Spark Version: {spark.version}")
print("SparkSession is running...")


Spark Version: 3.4.4
SparkSession is running...


# Staging

## Extract

### api

In [21]:
df_staging_api = extract_api_milestones_spark(spark, table_name='milestones')

+-------+----------+-------+------+----------+-------------------+---------+
|   step|   process| status|source|table_name|           etl_date|error_msg|
+-------+----------+-------+------+----------+-------------------+---------+
|staging|extraction|success|   api|milestones|2025-04-27 23:24:35|     null|
+-------+----------+-------+------+----------+-------------------+---------+



### spreadsheet

In [19]:
%%time
# relationship 
people_df = extract_spreadsheet_pyspark(spark,table_name='people')

# people 
relationships_df = extract_spreadsheet_pyspark(spark,table_name='relationships')

+-------+----------+-------+-----------+----------+-------------------+---------+
|   step|   process| status|     source|table_name|           etl_date|error_msg|
+-------+----------+-------+-----------+----------+-------------------+---------+
|staging|extraction|success|spreadsheet|    people|2025-04-27 16:29:44|     null|
+-------+----------+-------+-----------+----------+-------------------+---------+

+-------+----------+-------+-----------+-------------+-------------------+---------+
|   step|   process| status|     source|   table_name|           etl_date|error_msg|
+-------+----------+-------+-----------+-------------+-------------------+---------+
|staging|extraction|success|spreadsheet|relationships|2025-04-27 16:30:38|     null|
+-------+----------+-------+-----------+-------------+-------------------+---------+

CPU times: total: 3.64 s
Wall time: 1min 51s


### DB

In [20]:
%%time
# acquisition
acquisition = extract_database_pyspark(spark,'acquisition')

#company
company = extract_database_pyspark(spark,'company')

#funding_rounds
funding_rounds = extract_database_pyspark(spark,'funding_rounds')

#funds
funds = extract_database_pyspark(spark,'funds')

#investments
investments = extract_database_pyspark(spark,'investments')

#ipos
ipos = extract_database_pyspark(spark,'ipos')


+-------+----------+-------+--------+-----------+-------------------+---------+
|   step|   process| status|  source| table_name|           etl_date|error_msg|
+-------+----------+-------+--------+-----------+-------------------+---------+
|staging|extraction|success|database|acquisition|2025-04-27 16:31:36|     null|
+-------+----------+-------+--------+-----------+-------------------+---------+

+-------+----------+-------+--------+----------+-------------------+---------+
|   step|   process| status|  source|table_name|           etl_date|error_msg|
+-------+----------+-------+--------+----------+-------------------+---------+
|staging|extraction|success|database|   company|2025-04-27 16:32:09|     null|
+-------+----------+-------+--------+----------+-------------------+---------+

+-------+----------+-------+--------+--------------+-------------------+---------+
|   step|   process| status|  source|    table_name|           etl_date|error_msg|
+-------+----------+-------+--------+

## Load

### spreadsheet

In [21]:
load_staging_pyspark_upsert(spark, data=people_df, schema='public', table_name='people', idx_name='people_id', source='spreadsheet')
load_staging_pyspark_upsert(spark, data=relationships_df, schema='public', table_name='relationships', idx_name='relationship_id', source='spreadsheet')


+-------+-------+-------+-----------+----------+--------------------+---------+
|   step|process| status|     source|table_name|            etl_date|error_msg|
+-------+-------+-------+-----------+----------+--------------------+---------+
|staging|   load|success|spreadsheet|    people|2025-04-27 16:34:...|     null|
+-------+-------+-------+-----------+----------+--------------------+---------+

+-------+-------+-------+-----------+-------------+--------------------+---------+
|   step|process| status|     source|   table_name|            etl_date|error_msg|
+-------+-------+-------+-----------+-------------+--------------------+---------+
|staging|   load|success|spreadsheet|relationships|2025-04-27 16:35:...|     null|
+-------+-------+-------+-----------+-------------+--------------------+---------+



### api

In [22]:
load_staging_pyspark_upsert(spark, data=df_staging_api, schema='public', table_name='milestones', idx_name='milestone_id', source='api')


+-------+-------+-------+------+----------+--------------------+---------+
|   step|process| status|source|table_name|            etl_date|error_msg|
+-------+-------+-------+------+----------+--------------------+---------+
|staging|   load|success|   api|milestones|2025-04-27 23:43:...|     null|
+-------+-------+-------+------+----------+--------------------+---------+



### DB

In [22]:
# acquisition
load_staging_pyspark_upsert(spark, data=acquisition, schema='public', table_name='acquisition', idx_name='acquisition_id', source='database')
#company
load_staging_pyspark_upsert(spark, data=company, schema='public', table_name='company', idx_name='object_id', source='database')

#funding_rounds
load_staging_pyspark_upsert(spark, data=funding_rounds, schema='public', table_name='funding_rounds', idx_name='funding_round_id', source='database')

#funds
load_staging_pyspark_upsert(spark, data=funds, schema='public', table_name='funds', idx_name='fund_id', source='database')

#investments
load_staging_pyspark_upsert(spark, data=investments, schema='public', table_name='investments', idx_name='investment_id', source='database')

#ipos
load_staging_pyspark_upsert(spark, data=ipos, schema='public', table_name='ipos', idx_name='ipo_id', source='database')


+-------+-------+-------+--------+-----------+--------------------+---------+
|   step|process| status|  source| table_name|            etl_date|error_msg|
+-------+-------+-------+--------+-----------+--------------------+---------+
|staging|   load|success|database|acquisition|2025-04-27 16:36:...|     null|
+-------+-------+-------+--------+-----------+--------------------+---------+

+-------+-------+-------+--------+----------+--------------------+---------+
|   step|process| status|  source|table_name|            etl_date|error_msg|
+-------+-------+-------+--------+----------+--------------------+---------+
|staging|   load|success|database|   company|2025-04-27 16:37:...|     null|
+-------+-------+-------+--------+----------+--------------------+---------+

+-------+-------+-------+--------+--------------+--------------------+---------+
|   step|process| status|  source|    table_name|            etl_date|error_msg|
+-------+-------+-------+--------+--------------+------------

# Warehouse

## people

In [5]:
%%time
## people
 
# extract
people_staging = extract_db_pyspark_staging(spark,table_name='people')

# transform
dim_people = transform_dim_people_spark(spark, people_staging,'people')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_people, table_name='dim_people', schema='public', 
               idx_name='people_nk', source='staging',table_process='people')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|    people|2025-05-03 20:42:06|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|    people|2025-05-03 20:42:47|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## company


In [6]:
## company
# extract
company_staging = extract_db_pyspark_staging(spark,'company')
# transform
dim_company = transform_dim_company_spark(spark,company_staging,'company')
# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_company, table_name='dim_company', schema='public', 
               idx_name='company_nk', source='staging',table_process='company')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|   company|2025-05-03 20:44:12|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|   company|2025-05-03 20:44:47|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## relationships


In [7]:
# relationships 
relationships_staging = extract_db_pyspark_staging(spark,table_name='relationships')

# transform
dim_relationships = transform_dim_relationship_spark(spark,relationships_staging,'relationships')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_relationships, table_name='dim_relationships', schema='public', 
               idx_name='relationship_nk', source='staging',table_process='relationships')

+---------+----------+-------+--------+-------------+-------------------+---------+
|     step|   process| status|  source|   table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-------------+-------------------+---------+
|warehouse|extraction|success|database|relationships|2025-05-03 20:45:59|     null|
+---------+----------+-------+--------+-------------+-------------------+---------+

+---------+--------------+-------+-------+-------------+-------------------+---------+
|     step|       process| status| source|   table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+-------------+-------------------+---------+
|warehouse|transformation|success|staging|relationships|2025-05-03 20:46:30|     null|
+---------+--------------+-------+-------+-------------+-------------------+---------+

+---------+-------+-------+-------+-------------+--------------------+---------+
|     step|process| status| source|   table_name|            e

## funding_rounds

In [8]:
# funding_rounds 
funding_rounds_staging = extract_db_pyspark_staging(spark,table_name='funding_rounds')

# transform
fact_funding_rounds = transform_fact_funding_rounds_spark(spark,funding_rounds_staging,'funding_rounds')

# load
load_warehouse_pyspark_upsert(spark=spark,data=fact_funding_rounds, table_name='fact_funding_rounds', schema='public', 
               idx_name='funding_round_nk', source='staging',table_process='funding_rounds')

+---------+----------+-------+--------+--------------+-------------------+---------+
|     step|   process| status|  source|    table_name|           etl_date|error_msg|
+---------+----------+-------+--------+--------------+-------------------+---------+
|warehouse|extraction|success|database|funding_rounds|2025-05-03 20:47:51|     null|
+---------+----------+-------+--------+--------------+-------------------+---------+

+---------+--------------+-------+-------+--------------+-------------------+---------+
|     step|       process| status| source|    table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+--------------+-------------------+---------+
|warehouse|transformation|success|staging|funding_rounds|2025-05-03 20:48:23|     null|
+---------+--------------+-------+-------+--------------+-------------------+---------+

+---------+-------+-------+-------+--------------+--------------------+---------+
|     step|process| status| source|    table_name| 

## funds


In [9]:
# funds 
funds_staging = extract_db_pyspark_staging(spark,table_name='funds')

# transform
dim_funds = transform_fact_funds_spark(spark,funds_staging,'funds')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_funds, table_name='fact_funds', schema='public', 
               idx_name='fund_nk', source='staging',table_process='funds')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|     funds|2025-05-03 20:49:29|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|     funds|2025-05-03 20:50:03|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## ipos


In [10]:
# ipos 
ipos_staging = extract_db_pyspark_staging(spark,table_name='ipos')

# transform
dim_ipos = transform_fact_ipos_spark(spark,ipos_staging,'ipos')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_ipos, table_name='fact_ipos', schema='public', 
               idx_name='ipo_nk', source='staging',table_process='ipos')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|      ipos|2025-05-03 20:51:19|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|      ipos|2025-05-03 20:51:52|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------

## acquisitions

In [3]:
# ipos 
acquisitions_staging = extract_db_pyspark_staging(spark,table_name='acquisition')

# transform
dim_acquisition = transform_fact_acquisitions_spark(spark,acquisitions_staging,'acquisition')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_acquisition, table_name='fact_acquisition', schema='public', 
               idx_name='acquisition_nk', source='staging',table_process='acquisition')

+---------+----------+-------+--------+-----------+-------------------+---------+
|     step|   process| status|  source| table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-----------+-------------------+---------+
|warehouse|extraction|success|database|acquisition|2025-05-03 21:29:07|     null|
+---------+----------+-------+--------+-----------+-------------------+---------+

+---------+--------------+-------+-------+-----------+-------------------+---------+
|     step|       process| status| source| table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+-----------+-------------------+---------+
|warehouse|transformation|success|staging|acquisition|2025-05-03 21:29:35|     null|
+---------+--------------+-------+-------+-----------+-------------------+---------+

+---------+-------+-------+-------+-----------+--------------------+---------+
|     step|process| status| source| table_name|            etl_date|error_msg|
+----

## investments

In [12]:
# ipos 
investments_staging = extract_db_pyspark_staging(spark,'investments')

# transform
dim_investments = transform_fact_investments_spark(spark,investments_staging,'investments')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_investments, table_name='fact_investments', schema='public', 
               idx_name='investment_nk', source='staging',table_process='investments')

+---------+----------+-------+--------+-----------+-------------------+---------+
|     step|   process| status|  source| table_name|           etl_date|error_msg|
+---------+----------+-------+--------+-----------+-------------------+---------+
|warehouse|extraction|success|database|investments|2025-05-03 20:54:50|     null|
+---------+----------+-------+--------+-----------+-------------------+---------+

+---------+--------------+-------+-------+-----------+-------------------+---------+
|     step|       process| status| source| table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+-----------+-------------------+---------+
|warehouse|transformation|success|staging|investments|2025-05-03 20:55:25|     null|
+---------+--------------+-------+-------+-----------+-------------------+---------+

+---------+-------+-------+-------+-----------+--------------------+---------+
|     step|process| status| source| table_name|            etl_date|error_msg|
+----

In [15]:
dim_milestones.show()

+------------+------------+--------------------+--------------+----------+
|milestone_nk|milestone_at|         description|milestone_code|company_id|
+------------+------------+--------------------+--------------+----------+
|         100|  2009-12-15|Kenshoo Accounces...|         other|     17444|
|       10000|  2010-06-30|Metaconomy Hires ...|         other|     62216|
|       10001|  2011-01-19|DVDVideoSoft Prog...|         other|     72018|
|       10009|  2011-01-20|expanding into on...|         other|     56915|
|       10010|  2011-01-20|has named to its ...|         other|     61354|
|       10011|  2011-01-20|has won priority ...|         other|     57708|
|       10012|  2011-01-24|has pulled in a $...|         other|     59429|
|       10014|  2011-01-17|Winner of 2010 Ec...|         other|     62235|
|       10038|  2011-01-24|has received U.S....|         other|     78585|
|       10039|  2011-01-24|Schachter, Ex-Moo...|         other|     94165|
|        1004|  2008-04-1

## milestones

In [3]:
# ipos 
milestones_staging = extract_db_pyspark_staging(spark,'milestones')

# transform
dim_milestones = transform_fact_milestones_spark(spark,milestones_staging,'milestones')

# load
load_warehouse_pyspark_upsert(spark=spark,data=dim_milestones, table_name='fact_milestones', schema='public', 
               idx_name='milestone_nk', source='staging',table_process='milestones')

+---------+----------+-------+--------+----------+-------------------+---------+
|     step|   process| status|  source|table_name|           etl_date|error_msg|
+---------+----------+-------+--------+----------+-------------------+---------+
|warehouse|extraction|success|database|milestones|2025-05-03 21:17:37|     null|
+---------+----------+-------+--------+----------+-------------------+---------+

+---------+--------------+-------+-------+----------+-------------------+---------+
|     step|       process| status| source|table_name|           etl_date|error_msg|
+---------+--------------+-------+-------+----------+-------------------+---------+
|warehouse|transformation|success|staging|milestones|2025-05-03 21:18:04|     null|
+---------+--------------+-------+-------+----------+-------------------+---------+

+---------+-------+-------+-------+----------+--------------------+---------+
|     step|process| status| source|table_name|            etl_date|error_msg|
+---------+------